# V-Energy Grid 360 | Exploração da Gold — Fila de perdas V10

Notebook **somente leitura**, fora do Job. Chama o principal com `modo_exploracao=true`, calcula a V10 em memória e compara com a Gold publicada. Audita flags A1, inspeções A2, score/classe A3, snapshots A4, trocas A5 e as duas ações operacionais A6. Backtest opcional e observacional. **Não publicar outputs no GitHub.**


In [0]:
%run ./gold_fila_perdas $modo_exploracao="true" $publicar_gold="false" $homologacao_regras="false"


## 1. Controles da exploração

- `executar_backtest=false`: não calcula cortes históricos, economizando cota do Serverless.
- `mostrar_ucs=false`: resultados somente agregados por padrão; nunca subir outputs.
- `versao_delta_baseline` opcional: número da **versão Delta da tabela da fila** para comparação histórica (não é V6/V10 da regra). Vazio = publicação atual.

A lógica de publicação está bloqueada no notebook principal quando chamada neste modo.


**Como comparar apenas a regra (A4):** confira `versoes_silver_json` da publicação anterior, impresso na seção de snapshot. Se diferir da V10, configure `versoes_silver_fixas_json` no notebook principal com esse JSON e recalcule a proposta sem publicação. Esse controle usa Delta Time Travel e pode falhar quando versões antigas não estiverem mais disponíveis.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import re

dbutils.widgets.dropdown("executar_backtest", "false", ["false", "true"])
dbutils.widgets.dropdown("mostrar_ucs", "false", ["false", "true"])
dbutils.widgets.text("versao_delta_baseline", "")
EXECUTAR_BACKTEST = dbutils.widgets.get("executar_backtest") == "true"
MOSTRAR_UCS = dbutils.widgets.get("mostrar_ucs") == "true"
VERSAO_BASELINE = dbutils.widgets.get("versao_delta_baseline").strip()
TOPS_BACKTEST = (50, 200)
HORIZONTE_BACKTEST_MESES = 3
PASSO_BACKTEST_MESES = 3
MAX_CORTES_BACKTEST = 4
MIN_ROTULOS_COMPARACAO = 10

assert MODO_EXPLORACAO and not PUBLICAR_GOLD and not PUBLICACAO_EFETIVADA, "Proteção somente leitura não foi ativada: interrompendo."
assert (not VERSAO_BASELINE) or re.fullmatch(r"\d+", VERSAO_BASELINE), "Informe apenas o número da versão Delta."

NOME_FILA_PUBLICADA = f"{GOLD}.tb_fila_perdas"
NOME_AUDITORIA_PUBLICADA = f"{GOLD}.tb_fila_perdas_auditoria"
EXISTE_BASELINE = spark.catalog.tableExists(NOME_FILA_PUBLICADA)
if EXISTE_BASELINE:
    if VERSAO_BASELINE:
        baseline = spark.sql(f"SELECT * FROM {NOME_FILA_PUBLICADA} VERSION AS OF {int(VERSAO_BASELINE)}")
    else:
        baseline = spark.table(NOME_FILA_PUBLICADA)
    print("Baseline:", NOME_FILA_PUBLICADA, "Delta versão", VERSAO_BASELINE or "atual")
else:
    baseline = None
    print("Sem publicação anterior de tb_fila_perdas: serão exibidas apenas métricas da V10 em memória.")
print("V10 calculada em memória (modo exploração, sem publicação):", VERSAO_REGRA, "/ id_execucao:", ID_EXECUCAO)

# A4: uma comparação só permite atribuir diferenças à regra se as fontes Delta coincidirem.
BASELINE_SNAPSHOT = None
COMPARACAO_MESMO_SNAPSHOT = False
if baseline is not None and "versoes_silver_json" in baseline.columns:
    metadados = baseline.select("versoes_silver_json").distinct().limit(3).collect()
    if len(metadados) == 1:
        BASELINE_SNAPSHOT = metadados[0][0]
        try:
            COMPARACAO_MESMO_SNAPSHOT = json.loads(BASELINE_SNAPSHOT) == VERSOES_FONTES_INICIO
        except (ValueError, TypeError):
            COMPARACAO_MESMO_SNAPSHOT = False
print("A4 snapshot baseline:", BASELINE_SNAPSHOT or "NÃO DISPONÍVEL")
print("A4 snapshot V10:", VERSOES_FONTES_JSON)
print("A4 comparabilidade por versões Silver:",
      "MESMO_SNAPSHOT" if COMPARACAO_MESMO_SNAPSHOT else "NÃO COMPROVADA; NÃO ATRIBUIR DIFERENÇA APENAS À HEURÍSTICA")


## 2. Diagnóstico geral: V10 versus publicação anterior

A capacidade operacional é de **200 inspeções por mês**. A comparação considera quais UCs já haviam sido inspecionadas até o mês de referência e como as vagas se distribuem por classe. Os rótulos de inspeções anteriores **não** medem desempenho preditivo da V10.


In [0]:
print("Status de triagem V10 (universo total):")
display(universo_atual.groupBy("status_triagem", "confianca_evidencia")
    .agg(F.count("*").alias("ucs"), F.round(F.avg("score"),2).alias("score_medio"))
    .orderBy("status_triagem", "confianca_evidencia"))

print("Distribuição V10 dos elegíveis por faixa de score:")
faixas = (fila.withColumn("faixa_score", F.when(F.col("score") >= 80, "80-100")
    .when(F.col("score") >= 60,"60-79").when(F.col("score") >= 40,"40-59").otherwise("20-39")))
display(faixas.groupBy("faixa_score").agg(F.count("*").alias("ucs"),
    F.round(F.avg("pares_observados"),2).alias("pares_observados_medios"),
    F.round(F.sum("kwh_estimado"),2).alias("kwh_proxy_total")).orderBy("faixa_score"))

ids_inspecionadas_ate_ref = (insp_base
    .where(F.col("data_inspecao").isNotNull() & (F.col("data_inspecao") <= F.last_day(F.lit(referencia))))
    .select("id_uc").distinct().withColumn("inspecionada_antes",F.lit(True)))

def top_200(df, rotulo):
    if "posicao_fila" in df.columns:
        base_top = df.where(F.col("posicao_fila") <= TOP_N)
    else:
        w = Window.orderBy(F.desc("score"),F.desc("kwh_estimado"),F.asc("id_uc"))
        base_top = df.withColumn("posicao_fila",F.row_number().over(w)).where(F.col("posicao_fila") <= TOP_N)
    return (base_top.select("id_uc","classe","score","queda_pct","kwh_estimado","posicao_fila")
        .withColumn("cenario", F.lit(rotulo)))

tops = top_200(fila, "V10_CALCULADA")
if baseline is not None:
    faltam = sorted(set(("id_uc","classe","score","queda_pct","kwh_estimado")) - set(baseline.columns))
    if faltam:
        raise ValueError(f"Baseline sem colunas para comparar: {faltam}")
    tops = tops.unionByName(top_200(baseline,"PUBLICADA"))
resumo_tops = (tops.join(ids_inspecionadas_ate_ref,"id_uc","left")
    .withColumn("inspecionada_antes",F.coalesce(F.col("inspecionada_antes"),F.lit(False))))
display(resumo_tops.groupBy("cenario","classe").agg(
    F.count("*").alias("vagas_top_200"),
    F.sum(F.col("inspecionada_antes").cast("long")).alias("ja_inspecionadas"),
    F.round(F.avg(F.col("queda_pct")*100),1).alias("queda_pct_media"),
    F.round(F.avg("score"),2).alias("score_medio"))
    .orderBy("cenario",F.desc("vagas_top_200")))

print("Totais por cenário:")
display(resumo_tops.groupBy("cenario").agg(
    F.count("*").alias("total_top"),
    F.sum(F.col("inspecionada_antes").cast("long")).alias("ja_inspecionadas"),
    F.sum((~F.col("inspecionada_antes")).cast("long")).alias("novas")))

## 3. A1 — Arredondamento tolerado e negativos

Nem toda `flag_consumo_inconsistente` representa arredondamento. A V10 compara
consumo e diferença de leituras: até 1 kWh é tolerado; divergências materiais são
cruzadas com negativos. Negativos inválidos permanecem na Silver e são excluídos
dos cálculos Gold. Divergência material não negativa ou impossibilidade de comparação
bloqueia o principal para revisão na origem, preservando a regra existente.


In [0]:
print("Flags na Silver de leituras (sem IDs), reconciliadas na Gold V10:")
print("Contagens auditadas A1:", resumo_flags_a1)
display(leitura_auditada.where(F.col("__flag_inc"))
    .groupBy("__divergencia_material", "__negativo", "__flag_neg")
    .agg(F.count("*").alias("leituras"))
    .orderBy("__divergencia_material", "__negativo", "__flag_neg"))
display(leituras.agg(
    F.count("*").alias("leituras_total"),
    F.sum(F.col("flag_consumo_inconsistente").cast("long")).alias("flag_arredondamento"),
    F.sum(F.col("flag_consumo_negativo").cast("long")).alias("flag_negativo"),
    F.sum((F.col("consumo_kwh") < 0).cast("long")).alias("valor_negativo")))

if set(("leitura_anterior","leitura_atual")).issubset(leituras.columns):
    dif = F.abs(F.col("consumo_kwh") - (F.col("leitura_atual") - F.col("leitura_anterior")))
    display(leituras.where(F.col("flag_consumo_inconsistente"))
        .agg(F.count("*").alias("flagadas"),
             F.sum((dif > 1).cast("long")).alias("diferenca_maior_1_kwh"),
             F.sum(dif.isNull().cast("long")).alias("sem_comparacao")))

print("Efeito da V10 no universo (informativo, não fator de bloqueio):")
display(universo_atual.agg(
    F.sum(F.col("tem_arredondamento_pareado").cast("long")).alias("ucs_com_arredondamento_pareado"),
    F.sum((F.col("tem_arredondamento_pareado") & (F.col("status_triagem") == "ELEGIVEL")).cast("long")).alias("elegiveis_com_arredondamento")))

if spark.catalog.tableExists(NOME_AUDITORIA_PUBLICADA):
    aud_publicada = spark.table(NOME_AUDITORIA_PUBLICADA)
    if "fatores_atenuantes" in aud_publicada.columns:
        display(aud_publicada.agg(F.sum(F.array_contains("fatores_atenuantes", "CONSUMO_CORRIGIDO_OU_INCONSISTENTE").cast("long"))
            .alias("ucs_com_retencao_arredondamento_na_publicacao_atual")))
        print("Nota: auditoria acima é a tabela ATUAL; não confundir com uma baseline Delta histórica selecionada.")


## 4. A2 — Inspeções anteriores e risco de revisita

A janela proposta é de 12 meses. O notebook principal atribui status de revisão às UCs recentemente inspecionadas; esta seção verifica seu efeito em categorias e no top 200. Uma UC com inspeção antiga pode continuar elegível — decisão de revisita é de negócio.


In [0]:
display(universo_atual.groupBy("status_triagem").agg(
    F.count("*").alias("ucs"),
    F.sum(F.col("ja_inspecionada").cast("long")).alias("ja_inspecionadas"),
    F.sum(F.col("inspecao_recente").cast("long")).alias("inspecao_12_meses"))
    .orderBy("status_triagem"))

print("Inspeções históricas no top 200: veja resumo por cenário na seção 2.")
print("Não inferir taxa de acerto a partir de UCs já inspecionadas.")


## 5. A3 — Porte/classe e critérios de queda

Verifica a distribuição por classe e a condição de que o motivo de queda absoluta somente aparece com **queda percentual mínima de 20%** na proposta V10. O limiar é provisório, para homologar com a equipe.


In [0]:
display(fila.groupBy("classe").agg(F.count("*").alias("elegiveis"),
    F.round(F.avg(F.col("queda_pct")*100),1).alias("queda_pct_media"),
    F.round(F.avg("queda_kwh_mensal"),1).alias("queda_kwh_media"),
    F.round(F.avg("score"),1).alias("score_medio"))
    .orderBy(F.desc("elegiveis")))

violacoes = universo_atual.where(F.array_contains("motivos","QUEDA_ABSOLUTA_RELEVANTE") &
    (F.col("queda_pct") < LIMIAR_PCT_PARA_QUEDA_ABSOLUTA)).count()
assert violacoes == 0, "Motivo de queda absoluta atribuído sem queda percentual mínima."
print("Regressão A3 aprovada; motivos absolutos com queda inferior ao limiar:", violacoes)

print("A3 — Decomposição do score no TOP 200 (por classe):")
display(fila.where(F.col("posicao_fila") <= TOP_N)
    .groupBy("classe")
    .agg(F.count("*").alias("vagas"),
         F.round(F.avg(F.col("queda_pct") * 100), 2).alias("queda_pct_media"),
         F.round(F.avg("p_queda_pct"), 2).alias("p_queda_pct_media"),
         F.round(F.avg("p_queda_kwh"), 2).alias("p_queda_kwh_media"),
         F.round(F.avg("p_estimada"), 2).alias("p_estimada_media"),
         F.sum((F.col("queda_pct") < LIMIAR_PCT_PARA_QUEDA_ABSOLUTA).cast("long")).alias("vagas_com_queda_abaixo_limiar"))
    .orderBy(F.desc("vagas")))
print("A3 — Quantas indústrias com queda baixa ainda entram por sinais NÃO absolutos?")
display(fila.filter((F.col("classe") == "INDUSTRIAL") &
                    (F.col("queda_pct") < LIMIAR_PCT_PARA_QUEDA_ABSOLUTA))
    .groupBy("motivos_json")
    .agg(F.count("*").alias("ucs"))
    .orderBy(F.desc("ucs")))


## 5.1. A6 — Ação recomendada por evidência

A fila preserva elegíveis com `score >= 20` que superaram os controles de qualidade, mas **não confunde leitura estimada com perda comprovada**.

- `INSPECAO_PRIORITARIA`: a UC tem queda sazonal ou absoluta relevante.
- `VALIDACAO_DOCUMENTAL`: queda independente insuficiente; estimadas recorrentes justificam revisão de medição **antes** de eventual visita.

`posicao_fila` coloca primeiro a faixa de inspeção prioritária, depois a documental. O número 200 representa capacidade mensal, não meta de ocupação. **A exploração não gera ordens de inspeção.**


In [0]:
print("A6 — composição da fila por tipo de ação e classe")
display(fila.groupBy("acao_recomendada", "classe").agg(
    F.count("*").alias("ucs"),
    F.round(F.avg(F.col("queda_pct") * 100), 2).alias("queda_pct_media"),
    F.round(F.avg("score"), 2).alias("score_medio"),
    F.round(F.sum("kwh_estimado"), 2).alias("kwh_proxy"))
    .orderBy("acao_recomendada", F.desc("ucs")))

print("A6 — totais por ação no ranking")
display(fila.groupBy("prioridade_operacional", "acao_recomendada")
    .agg(F.count("*").alias("ucs"),
         F.min("posicao_fila").alias("primeira_posicao"),
         F.max("posicao_fila").alias("ultima_posicao"))
    .orderBy("prioridade_operacional"))

assert fila.filter((F.col("acao_recomendada") == "INSPECAO_PRIORITARIA") &
                   (~F.col("queda_qualificada"))).limit(1).count() == 0
assert fila.filter((F.col("acao_recomendada") == "VALIDACAO_DOCUMENTAL") &
                   (F.col("queda_qualificada") |
                    ~F.array_contains("motivos", "LEITURAS_ESTIMADAS_RECORRENTES"))).limit(1).count() == 0
n_elegiveis = fila.count()
assert n_elegiveis == sum(resumo_operacional.values())
print(f"Elegíveis={n_elegiveis}; limite de capacidade={TOP_N}; vagas não ocupadas={max(0, TOP_N-n_elegiveis)}")
print("Ação documental requer checagem anterior à ordem de campo; nenhuma evidência adicional foi inventada.")


## 6. A4 e A5 — Publicação, trocas e rastreabilidade

A4: versões publicadas não devem ser confundidas com o código em Git. A5: avaliar o efeito da janela completa de trocas. Usar identificador de execução para evitar comparar snapshots incompatíveis.

**Contrato da V10:** os sinais intermediários `troca_sinalizada_base` e `troca_sinalizada_silver` são usados no cálculo interno, mas não são publicados por `selecionar_colunas_gold`. A exploração usa a flag consolidada `houve_troca_na_janela` para medir o efeito da regra, sem modificar o contrato da Gold. Para detalhar os tipos de troca, seria necessária uma consulta específica às leituras pareadas, e não uma referência a colunas ausentes.


In [0]:
print("Snapshot V10 em memória:", VERSAO_REGRA, ID_EXECUCAO, "Silver:", VERSOES_FONTES_JSON)
if baseline is not None:
    for c in ("versao_regra", "id_execucao", "mes_referencia"):
        if c in baseline.columns:
            display(baseline.groupBy(c).agg(F.count("*").alias("ucs")))

print("Indicadores de troca na janela-base/atual (V10):")
# Os dois sinais parciais existem no cálculo interno da Gold, mas não no
# DataFrame final. A exploração lê somente o contrato publicado pela V10.
assert "houve_troca_na_janela" in universo_atual.columns, (
    "V10 sem houve_troca_na_janela: confira o notebook principal e seu contrato."
)
display(universo_atual.groupBy("status_triagem").agg(
    F.count("*").alias("ucs"),
    F.sum(F.col("houve_troca_na_janela").cast("long")).alias("ucs_com_troca_na_janela"))
    .orderBy("status_triagem"))

print("Concentração geográfica de elegíveis (resumo agregado):")
display(fila.groupBy("municipio","alimentador").agg(
    F.count("*").alias("ucs"),F.round(F.sum("kwh_estimado"),2).alias("kwh_proxy"))
    .orderBy(F.desc("ucs")).limit(15))

print("A5 — retenções por troca nas UCs com indícios de queda significativa:")
display(universo_atual.where(F.col("historico_suficiente") &
        (F.col("queda_pct") >= LIMIAR_PCT_PARA_QUEDA_ABSOLUTA))
    .groupBy("houve_troca_na_janela", "status_triagem")
    .agg(F.count("*").alias("ucs"), F.round(F.avg("score"),2).alias("score_medio"))
    .orderBy("houve_troca_na_janela", "status_triagem"))
print("A5: presença de troca não demonstra automaticamente leitura inválida; manter revisão humana.")


## 7. Inspeção de amostras (opcional, confidencial)

Somente com `mostrar_ucs=true` — nunca commitar outputs. Os resultados abaixo são identificáveis e devem ficar restritos ao workspace autorizado.


In [0]:
if MOSTRAR_UCS:
    print("Top 20 em memória (V10) por score:")
    display(fila.orderBy(F.asc("posicao_fila"))
        .select("posicao_fila","id_uc","classe","municipio","bairro","score",
                "confianca_evidencia","motivos","fatores_atenuantes","kwh_estimado").limit(20))
    print("Top 20 por proxy de kWh (objetivo diferente de maior suspeição):")
    display(fila.orderBy(F.desc("kwh_estimado"),F.desc("score"),F.asc("id_uc"))
        .select("id_uc","classe","score","kwh_estimado","queda_pct").limit(20))
else:
    print("Amostras individuais desabilitadas. Somente os agregados foram exibidos.")


## 8. Backtest histórico — opcional e observacional

Este backtest foi **movido integralmente do notebook principal**, sem reescrever sua função de cálculo. Quando `executar_backtest=true`, ele chama `calcular_gold(ref_bt)` em cortes trimestrais, usa apenas históricos disponíveis até cada corte e exclui UCs previamente inspecionadas. Os resultados **não estimam precisão populacional**: apenas caracterizam as UCs inspecionadas posteriormente.


In [0]:
# Backtest exploratório: cortes cronológicos e primeira inspeção conclusiva após o corte.
# Não há amostragem aleatória; métricas observacionais, não precisão populacional.
if EXECUTAR_BACKTEST:
    from datetime import date

    def somar_meses(mes, n):
        indice = mes.year * 12 + mes.month - 1 + n
        return date(indice // 12, indice % 12 + 1, 1)

    resumo_datas = insp_base.where(F.col("data_inspecao").isNotNull()).agg(
        F.min("data_inspecao").alias("primeira"),
        F.max("data_inspecao").alias("ultima")).first()
    primeira = resumo_datas["primeira"]
    ultima = resumo_datas["ultima"]
    primeira_leitura = mensal.agg(F.min("mes").alias("minimo")).first()["minimo"]
    if primeira is None or primeira_leitura is None:
        print("Backtest indisponível: não há inspeções ou leituras datadas.")
    else:
        # Corte 1: mês anterior à primeira inspeção. Janela de avaliação: próximos 3 meses completos.
        corte_inicial = somar_meses(primeira.replace(day=1), -1)
        ultimo_corte = somar_meses(ultima.replace(day=1), -HORIZONTE_BACKTEST_MESES)
        cortes = []
        corte = corte_inicial
        while corte <= ultimo_corte and len(cortes) < MAX_CORTES_BACKTEST:
            meses_de_historico = (corte.year - primeira_leitura.year) * 12 + corte.month - primeira_leitura.month
            if meses_de_historico >= 12 + JANELA_MESES - 1:
                cortes.append(corte)
            corte = somar_meses(corte, PASSO_BACKTEST_MESES)

        if not cortes:
            print("Backtest indisponível: não há cortes com histórico suficiente e janela posterior completa.")
        else:
            metricas = []
            for ref_bt in cortes:
                corte_fim = F.last_day(F.lit(ref_bt).cast("date"))
                prox_inicio = F.date_add(corte_fim, 1)
                prox_fim = F.last_day(F.add_months(corte_fim, HORIZONTE_BACKTEST_MESES))
                # Uma UC pode ter múltiplas inspeções; usamos somente a primeira conclusiva.
                w_inspecao = Window.partitionBy("id_uc").orderBy(F.asc("data_inspecao"), F.asc("id_inspecao"))
                rotulos = (insp_base
                    .where(F.col("data_inspecao").between(prox_inicio, prox_fim))
                    .where(F.col("resultado").isin("IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE"))
                    .withColumn("ordem_inspecao", F.row_number().over(w_inspecao))
                    .where(F.col("ordem_inspecao") == 1)
                    .select("id_uc",
                        F.when(F.col("resultado") == "IRREGULARIDADE CONFIRMADA", 1).otherwise(0).alias("confirmada"),
                        F.when(F.col("resultado") == "IRREGULARIDADE CONFIRMADA",
                            F.when(F.col("kwh_recuperado").isNotNull() &
                                   (F.col("kwh_recuperado") >= 0), F.col("kwh_recuperado")))
                            .otherwise(F.lit(None).cast("double")).alias("kwh_recuperado_observado")))
                # A2: evitar reaproveitar casos já inspecionados ANTES do corte
                # como candidatos "novos" (sem vazamento dos rótulos futuros).
                ids_inspecionadas_antes_corte = (insp_base
                    .where(F.col("data_inspecao").isNotNull() & (F.col("data_inspecao") <= corte_fim))
                    .select("id_uc").distinct())
                candidatos_bt = (calcular_gold(ref_bt)
                    .where(F.col("status_triagem") == "ELEGIVEL")
                    .join(ids_inspecionadas_antes_corte, "id_uc", "left_anti"))
                w_ranking = Window.orderBy(F.asc("prioridade_operacional"), F.desc("score"), F.desc("kwh_estimado"), F.asc("id_uc"))
                classificados = candidatos_bt.withColumn("posicao", F.row_number().over(w_ranking))
                n_elegiveis = classificados.count()
                rotulados = classificados.join(rotulos, "id_uc", "inner")
                n_inspecionados = rotulados.count()
                for top_n in TOPS_BACKTEST:
                    n_top = min(top_n, n_elegiveis)
                    resultado = rotulados.agg(
                        F.sum(F.when(F.col("posicao") <= n_top, 1).otherwise(0)).alias("rotulos_top"),
                        F.sum(F.when((F.col("posicao") <= n_top) & (F.col("confirmada") == 1), 1).otherwise(0)).alias("confirmadas_top"),
                        F.sum(F.when(F.col("posicao") > n_top, 1).otherwise(0)).alias("rotulos_outros"),
                        F.sum(F.when((F.col("posicao") > n_top) & (F.col("confirmada") == 1), 1).otherwise(0)).alias("confirmadas_outros"),
                        F.sum(F.when(F.col("posicao") <= n_top,
                            F.col("kwh_recuperado_observado"))).alias("kwh_recuperado_top"),
                        F.count(F.when((F.col("posicao") <= n_top) &
                            F.col("kwh_recuperado_observado").isNotNull(), 1)).alias("rotulos_kwh_validos_top"),
                    ).first()
                    rtop = int(resultado["rotulos_top"] or 0)
                    ctop = int(resultado["confirmadas_top"] or 0)
                    routros = int(resultado["rotulos_outros"] or 0)
                    coutros = int(resultado["confirmadas_outros"] or 0)
                    taxa_top = ctop / rtop if rtop else None
                    taxa_outros = coutros / routros if routros else None
                    lift = taxa_top / taxa_outros if (taxa_top is not None and taxa_outros is not None and taxa_outros > 0) else None
                    if routros == 0 or rtop == 0:
                        status = "SEM_COMPARADOR_OBSERVADO"
                        lift = None
                    elif min(rtop, routros) < MIN_ROTULOS_COMPARACAO:
                        status = "AMOSTRA_OBSERVADA_PEQUENA"
                    else:
                        status = "COMPARACAO_OBSERVACIONAL"
                    metricas.append((str(ref_bt), int(top_n), n_elegiveis, int(n_top),
                        n_inspecionados, rtop, ctop, routros, coutros, taxa_top,
                        taxa_outros, lift, rtop/n_top if n_top else None,
                        (float(resultado["kwh_recuperado_top"])
                            if resultado["kwh_recuperado_top"] is not None else None),
                        int(resultado["rotulos_kwh_validos_top"] or 0), status))
                print(f"Corte {ref_bt}: elegíveis={n_elegiveis}, com inspeção observada={n_inspecionados}"
                      f" nos {HORIZONTE_BACKTEST_MESES} meses seguintes.")
            # Schema explícito evita inferência incorreta quando comparadores não têm rótulos.
            colunas_bt = ["competencia_corte", "top_planejado", "elegiveis_total", "top_real",
                "elegiveis_inspecionados", "top_inspecionados", "confirmadas_top",
                "outros_inspecionados", "confirmadas_outros", "taxa_top", "taxa_outros",
                "razao_taxas", "cobertura_rotulos_top", "kwh_recuperado_observado_top", "rotulos_kwh_validos_top", "qualidade_comparacao"]
            tipos_bt = "string, int, long, int, long, long, long, long, long, double, double, double, double, double, long, string"
            schema_bt = ", ".join(f"{col} {tipo}" for col,tipo in zip(colunas_bt,tipos_bt.split(", ")))
            display(spark.createDataFrame(metricas, schema=schema_bt).orderBy("competencia_corte", "top_planejado"))
            print("ALERTA: resultados condicionados à seleção das UCs que foram inspecionadas;")
            print("não extrapolar taxas ou kWh observados para toda a população.")
else:
    print("Backtest desligado para reduzir consumo do Serverless. Use executar_backtest=true para avaliação.")

## 9. Conferência final (sem escrita)

1. Rode a exploração com `executar_backtest=true` quando quiser avaliar o histórico observacional (a taxa não é causal).
2. Compare a V10 com a Gold publicada usando o mesmo snapshot Silver; se os snapshots divergirem, declare a limitação.
3. Confira os resultados A1 a A5, a composição do top 200, as faixas da A6 e os metadados de publicação.
4. Alinhe os limites propostos com Rafael e Théo, e o contrato da API com o Dev.
5. Se aprovado, rode o `gold_fila_perdas` **diretamente**, fora deste notebook, com `modo_exploracao=false`, `publicar_gold=true` e `homologacao_regras=true`; depois revise a Gold de novo.

A fila não impõe 200 candidatas nem transforma estimadas em fraude comprovada. Este notebook não entra no Job. Limpe todos os outputs antes do commit.